In [6]:
import os
import sys
import logging
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from tqdm.auto import tqdm

sys.path.insert(0, "../src")  # make validation/src importable
from build_amy_epw import AMYEPWBuilder

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

load_dotenv("../.env")
NREL_API_KEY = os.environ["NREL_API_KEY"]
NREL_EMAIL   = os.environ["NREL_EMAIL"]

YEAR    = 2023
OUT_DIR = Path("../src/data/weather")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load station_ids from quality-filtered valid sites



In [2]:
VALID_STATION_IDS_PATH = Path("../src/data/metadata/valid_site_station_ids.csv")

if not VALID_STATION_IDS_PATH.exists():
    raise FileNotFoundError(
        f"{VALID_STATION_IDS_PATH} does not exist. Run the site coverage map cell in proc.ipynb first."
    )


def normalize_station_id(station_id):
    station_id = str(station_id).strip()
    if "-" not in station_id:
        return station_id
    usaf, wban = station_id.split("-", 1)
    return f"{usaf}-{wban.zfill(5)}"

valid_station_ids_raw = (
    pd.read_csv(VALID_STATION_IDS_PATH, dtype={"station_id": str})["station_id"]
    .dropna()
    .unique()
)
valid_station_ids = pd.Index(valid_station_ids_raw).map(normalize_station_id).unique()
station_id_normalization = (
    pd.DataFrame({"station_id_raw": valid_station_ids_raw})
    .assign(station_id=lambda df: df["station_id_raw"].map(normalize_station_id))
)

print(f"Loaded {len(valid_station_ids)} unique station IDs from {VALID_STATION_IDS_PATH}")
print(valid_station_ids)

Loaded 54 unique station IDs from ..\src\data\metadata\valid_site_station_ids.csv
Index(['720322-04129', '720388-00469', '720638-00224', '720734-00264',
       '720749-24255', '720923-00310', '725785-24145', '725866-94178',
       '725895-94236', '725970-24225', '725975-24235', '726810-24131',
       '726813-94195', '726836-04201', '726881-94273', '726883-04113',
       '726904-24231', '726917-24284', '726920-24230', '726930-24221',
       '726945-24202', '726950-24285', '726959-94281', '726980-24229',
       '726985-24242', '726986-94261', '727730-24153', '727790-24146',
       '727810-24243', '727830-24149', '727834-24136', '727845-24163',
       '727846-24160', '727850-24157', '727856-94176', '727857-94129',
       '727870-94119', '727873-99999', '727883-24220', '727884-99999',
       '727918-94298', '727924-24223', '727925-94227', '727928-94263',
       '727930-24233', '727934-94248', '727935-24234', '727937-24222',
       '727938-94274', '727945-04205', '742060-24207', '994025-999

## Build DDY files from the nearest Climate.OneBuilding TMYx package

For EnergyPlus sizing, use the vetted design-day file bundled with the nearest TMYx weather ZIP instead of deriving DDY data from the AMY EPW. This cell resolves each valid WBAN station ID to NOAA ISD coordinates, finds the closest USA TMYx location from Climate.OneBuilding, downloads the ZIP, and extracts the included `.ddy` file next to the AMY `.epw`.

In [7]:
import re
import zipfile
from urllib.parse import urljoin, urlparse

import numpy as np
import requests

TMYX_USA_XLSX_URL = "https://climate.onebuilding.org/sources/Region4_USA_TMYx_EPW_Processing_locations.xlsx"
ISD_HISTORY_URL = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"
TMYX_BASE_URL = "https://climate.onebuilding.org/"
DDY_OUT_DIR = OUT_DIR
TMYX_CACHE_DIR = OUT_DIR / "tmyx_cache"
DDY_MATCH_SUMMARY_PATH = OUT_DIR / "tmyx_ddy_match_summary.csv"
TMYX_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def clean_col_name(value):
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")


def find_column(columns, candidates):
    normalized = {clean_col_name(col): col for col in columns}
    for candidate in candidates:
        key = clean_col_name(candidate)
        if key in normalized:
            return normalized[key]
    for col in columns:
        key = clean_col_name(col)
        if any(clean_col_name(candidate) in key for candidate in candidates):
            return col
    return None


def normalize_wban(value):
    text = str(value).strip()
    if "-" in text:
        text = text.split("-")[-1]
    text = re.sub(r"\D", "", text)
    return text.zfill(5) if text else pd.NA


def normalize_zip_url(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text or ".zip" not in text.lower():
        return pd.NA
    match = re.search(r"https?://[^\s,;]+?\.zip", text, flags=re.IGNORECASE)
    if match:
        return match.group(0)
    match = re.search(r"[^\s,;]+?\.zip", text, flags=re.IGNORECASE)
    if match:
        return urljoin(TMYX_BASE_URL, match.group(0))
    return pd.NA


def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))


print("Loading Climate.OneBuilding USA TMYx catalog...")
tmyx_catalog = pd.read_excel(TMYX_USA_XLSX_URL)
tmyx_catalog.columns = [str(col).strip() for col in tmyx_catalog.columns]

lat_col = find_column(tmyx_catalog.columns, ["latitude", "lat"])
lon_col = find_column(tmyx_catalog.columns, ["longitude", "lon", "long"])
name_col = find_column(tmyx_catalog.columns, ["location", "name", "station", "city"])

zip_col = None
for col in tmyx_catalog.columns:
    sample = tmyx_catalog[col].dropna().astype(str).head(200)
    if sample.str.contains(r"\.zip", case=False, regex=True).any():
        zip_col = col
        break

if lat_col is None or lon_col is None or zip_col is None:
    raise ValueError(
        "Could not identify latitude, longitude, and ZIP URL columns in the TMYx catalog. "
        f"Columns found: {list(tmyx_catalog.columns)}"
    )

tmyx_catalog = tmyx_catalog.copy()
tmyx_catalog["tmyx_lat"] = pd.to_numeric(tmyx_catalog[lat_col], errors="coerce")
tmyx_catalog["tmyx_lon"] = pd.to_numeric(tmyx_catalog[lon_col], errors="coerce")
tmyx_catalog["tmyx_zip_url"] = tmyx_catalog[zip_col].map(normalize_zip_url)
tmyx_catalog["tmyx_name"] = tmyx_catalog[name_col].astype(str) if name_col else ""
tmyx_catalog = tmyx_catalog.dropna(subset=["tmyx_lat", "tmyx_lon", "tmyx_zip_url"]).reset_index(drop=True)
print(f"Loaded {len(tmyx_catalog):,} usable USA TMYx locations")

print("Loading NOAA ISD station coordinates...")
isd = pd.read_csv(ISD_HISTORY_URL, dtype=str)
isd.columns = [str(col).strip().upper() for col in isd.columns]
isd["WBAN"] = isd["WBAN"].map(normalize_wban)
isd["station_lat"] = pd.to_numeric(isd["LAT"], errors="coerce")
isd["station_lon"] = pd.to_numeric(isd["LON"], errors="coerce")
isd["end_sort"] = pd.to_numeric(isd.get("END", pd.Series(index=isd.index)), errors="coerce").fillna(0)
isd_coords = (
    isd.dropna(subset=["WBAN", "station_lat", "station_lon"])
    .sort_values(["WBAN", "end_sort"], ascending=[True, False])
    .drop_duplicates("WBAN")
    [["WBAN", "station_lat", "station_lon"]]
)

station_coords = pd.DataFrame({"station_id": pd.Index(valid_station_ids).astype(str)})
station_coords["wban"] = station_coords["station_id"].map(normalize_wban)
station_coords = station_coords.merge(isd_coords, left_on="wban", right_on="WBAN", how="left")

records = []
for row in station_coords.itertuples(index=False):
    station_id = row.station_id
    ddy_path = DDY_OUT_DIR / f"{station_id}.ddy"

    if pd.isna(row.station_lat) or pd.isna(row.station_lon):
        records.append({
            "station_id": station_id,
            "station_lat": row.station_lat,
            "station_lon": row.station_lon,
            "status": "missing_noaa_isd_coordinates",
        })
        continue

    distances = haversine_km(row.station_lat, row.station_lon, tmyx_catalog["tmyx_lat"], tmyx_catalog["tmyx_lon"])
    nearest_idx = int(np.nanargmin(distances))
    nearest = tmyx_catalog.iloc[nearest_idx]
    zip_url = nearest["tmyx_zip_url"]
    zip_name = Path(urlparse(zip_url).path).name
    zip_path = TMYX_CACHE_DIR / zip_name

    try:
        if not zip_path.exists():
            response = requests.get(zip_url, timeout=90)
            response.raise_for_status()
            zip_path.write_bytes(response.content)

        with zipfile.ZipFile(zip_path) as zf:
            ddy_members = [name for name in zf.namelist() if name.lower().endswith(".ddy")]
            if not ddy_members:
                raise FileNotFoundError(f"No .ddy file found in {zip_name}")
            ashrae_members = [name for name in ddy_members if "ashrae" in name.lower()]
            ddy_member = ashrae_members[0] if ashrae_members else ddy_members[0]
            ddy_path.write_bytes(zf.read(ddy_member))

        status = "saved"
        error = ""
    except Exception as exc:
        status = "failed"
        error = str(exc)
        ddy_member = pd.NA

    records.append({
        "station_id": station_id,
        "station_lat": row.station_lat,
        "station_lon": row.station_lon,
        "tmyx_name": nearest.get("tmyx_name", ""),
        "tmyx_lat": nearest["tmyx_lat"],
        "tmyx_lon": nearest["tmyx_lon"],
        "distance_km": float(distances[nearest_idx]),
        "tmyx_zip_url": zip_url,
        "tmyx_zip_path": str(zip_path),
        "ddy_member": ddy_member,
        "ddy_path": str(ddy_path) if status == "saved" else "",
        "status": status,
        "error": error,
    })

tmyx_ddy_match_summary = pd.DataFrame(records)
tmyx_ddy_match_summary.to_csv(DDY_MATCH_SUMMARY_PATH, index=False)
print(f"Saved DDY match summary to {DDY_MATCH_SUMMARY_PATH}")
print(tmyx_ddy_match_summary["status"].value_counts(dropna=False))
display(tmyx_ddy_match_summary.sort_values(["status", "distance_km"], na_position="last"))

Loading Climate.OneBuilding USA TMYx catalog...
Loaded 13,356 usable USA TMYx locations
Loading NOAA ISD station coordinates...
Saved DDY match summary to ..\src\data\weather\tmyx_ddy_match_summary.csv
status
saved    54
Name: count, dtype: int64


,station_id,station_lat,station_lon,tmyx_name,tmyx_lat,tmyx_lon,distance_km,tmyx_zip_url,tmyx_zip_path,ddy_member,ddy_path,status,error
0,720322-04129,48.299,-116.560,Sandpoint.AP,48.29900,-116.5600,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_ID_Sandpoin...,USA_ID_Sandpoint.AP.720322_TMYx.2004-2018.ddy,..\src\data\weather\720322-04129.ddy,saved,
1,720388-00469,47.104,-122.287,Puyallup-Pierce.County.AP-Thun.Field,47.10400,-122.2870,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_WA_Puyallup...,USA_WA_Puyallup-Pierce.County.AP-Thun.Field.72...,..\src\data\weather\720388-00469.ddy,saved,
2,720638-00224,44.095,-121.200,Bend.Muni.AP,44.09500,-121.2000,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_OR_Bend.Mun...,USA_OR_Bend.Muni.AP.720638_TMYx.2004-2018.ddy,..\src\data\weather\720638-00224.ddy,saved,
3,720734-00264,43.581,-116.523,Nampa.Muni.AP,43.58100,-116.5230,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_ID_Nampa.Mu...,USA_ID_Nampa.Muni.AP.720734_TMYx.2004-2018.ddy,..\src\data\weather\720734-00264.ddy,saved,
4,720749-24255,48.350,-122.667,Oak.Harbor-NAS.Whidbey.Island-Ault.Field,48.35000,-122.6670,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_WA_Oak.Harb...,USA_WA_Oak.Harbor-NAS.Whidbey.Island-Ault.Fiel...,..\src\data\weather\720749-24255.ddy,saved,
5,720923-00310,48.726,-116.295,Boundary.County.AP,48.72600,-116.2950,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_ID_Boundary...,USA_ID_Boundary.County.AP.720923_TMYx.2004-201...,..\src\data\weather\720923-00310.ddy,saved,
11,726810-24131,43.567,-116.241,Boise.AP-Gowen.Field.ANGB,43.56700,-116.2410,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_ID_Boise.AP...,USA_ID_Boise.AP-Gowen.Field.ANGB.726810_TMYx.2...,..\src\data\weather\726810-24131.ddy,saved,
12,726813-94195,43.650,-116.633,Caldwell.Indl.AP,43.65000,-116.6330,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_ID_Caldwell...,USA_ID_Caldwell.Indl.AP.726813_TMYx.2007-2021.ddy,..\src\data\weather\726813-94195.ddy,saved,
15,726883-04113,45.826,-119.261,Hermiston.Muni.AP,45.82600,-119.2610,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_OR_Hermisto...,USA_OR_Hermiston.Muni.AP.726883_TMYx.ddy,..\src\data\weather\726883-04113.ddy,saved,
17,726917-24284,43.413,-124.244,North.Bend-Southwest.Oregon.Rgnl.AP,43.41300,-124.2440,0.000000,https://climate.onebuilding.org/WMO_Region_4_N...,..\src\data\weather\tmyx_cache\USA_OR_North.Be...,USA_OR_North.Bend-Southwest.Oregon.Rgnl.AP.726...,..\src\data\weather\726917-24284.ddy,saved,


## Build EPW files ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â one per station_id

In [3]:
errors = {}

for station_id in tqdm(valid_station_ids, desc="Building EPW files"):
    out_path = OUT_DIR / f"{station_id}.epw"
    legacy_station_ids = station_id_normalization.loc[
        station_id_normalization["station_id"] == station_id, "station_id_raw"
    ].tolist()
    legacy_out_paths = [OUT_DIR / f"{legacy_station_id}.epw" for legacy_station_id in legacy_station_ids]

    if out_path.exists():
        print(f"  SKIP  {station_id}  (already exists)")
        continue

    existing_legacy_path = next((path for path in legacy_out_paths if path.exists()), None)
    if existing_legacy_path is not None:
        existing_legacy_path.rename(out_path)
        print(f"  RENAME  {existing_legacy_path.name} -> {out_path.name}")
        continue

    try:
        builder = AMYEPWBuilder(station_id, YEAR, NREL_API_KEY, NREL_EMAIL)
        builder.build(str(out_path))
        print(f"  OK    {station_id} -> {out_path}")
    except Exception as exc:
        errors[station_id] = exc
        print(f"  FAIL  {station_id}: {exc}")

print(f"Done -- {len(valid_station_ids) - len(errors)} succeeded, {len(errors)} failed")
if errors:
    print("Failed station IDs:", list(errors.keys()))

Building EPW files: 100%|ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ | 54/54 [00:00<00:00, 1996.72it/s]

  SKIP  720322-04129  (already exists)
  SKIP  720388-00469  (already exists)
  SKIP  720638-00224  (already exists)
  SKIP  720734-00264  (already exists)
  SKIP  720749-24255  (already exists)
  SKIP  720923-00310  (already exists)
  SKIP  725785-24145  (already exists)
  SKIP  725866-94178  (already exists)
  SKIP  725895-94236  (already exists)
  SKIP  725970-24225  (already exists)
  SKIP  725975-24235  (already exists)
  SKIP  726810-24131  (already exists)
  SKIP  726813-94195  (already exists)
  SKIP  726836-04201  (already exists)
  SKIP  726881-94273  (already exists)
  SKIP  726883-04113  (already exists)
  SKIP  726904-24231  (already exists)
  SKIP  726917-24284  (already exists)
  SKIP  726920-24230  (already exists)
  SKIP  726930-24221  (already exists)
  SKIP  726945-24202  (already exists)
  SKIP  726950-24285  (already exists)
  SKIP  726959-94281  (already exists)
  SKIP  726980-24229  (already exists)
  SKIP  726985-24242  (already exists)
  SKIP  726986-94261  (al

## Package EPW and DDY files per station

Create one ZIP archive per valid station containing the matching `.epw` and `.ddy` files. The summary table flags any stations that are missing either file.

In [8]:
import zipfile

WEATHER_ZIP_DIR = OUT_DIR / "station_weather_zips"
WEATHER_ZIP_DIR.mkdir(parents=True, exist_ok=True)

weather_zip_records = []
for station_id in pd.Index(valid_station_ids).astype(str):
    epw_path = OUT_DIR / f"{station_id}.epw"
    ddy_path = OUT_DIR / f"{station_id}.ddy"
    zip_path = WEATHER_ZIP_DIR / f"{station_id}_weather.zip"

    missing_files = [path.name for path in [epw_path, ddy_path] if not path.exists()]
    if missing_files:
        weather_zip_records.append({
            "station_id": station_id,
            "zip_path": str(zip_path),
            "status": "missing_files",
            "missing_files": ", ".join(missing_files),
        })
        continue

    with zipfile.ZipFile(zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(epw_path, arcname=epw_path.name)
        zf.write(ddy_path, arcname=ddy_path.name)

    weather_zip_records.append({
        "station_id": station_id,
        "zip_path": str(zip_path),
        "status": "saved",
        "missing_files": "",
    })

weather_zip_summary = pd.DataFrame(weather_zip_records)
print(f"Saved {weather_zip_summary['status'].eq('saved').sum()} station weather ZIPs to {WEATHER_ZIP_DIR}")
print(weather_zip_summary["status"].value_counts(dropna=False))
display(weather_zip_summary.sort_values(["status", "station_id"]))

Saved 54 station weather ZIPs to ..\src\data\weather\station_weather_zips
status
saved    54
Name: count, dtype: int64


,station_id,zip_path,status,missing_files
0,720322-04129,..\src\data\weather\station_weather_zips\72032...,saved,
1,720388-00469,..\src\data\weather\station_weather_zips\72038...,saved,
2,720638-00224,..\src\data\weather\station_weather_zips\72063...,saved,
3,720734-00264,..\src\data\weather\station_weather_zips\72073...,saved,
4,720749-24255,..\src\data\weather\station_weather_zips\72074...,saved,
5,720923-00310,..\src\data\weather\station_weather_zips\72092...,saved,
6,725785-24145,..\src\data\weather\station_weather_zips\72578...,saved,
7,725866-94178,..\src\data\weather\station_weather_zips\72586...,saved,
8,725895-94236,..\src\data\weather\station_weather_zips\72589...,saved,
9,725970-24225,..\src\data\weather\station_weather_zips\72597...,saved,
